In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import os

# Paths
input_csv = "../data/competitions.csv"
output_csv = "../data/competition_logos.csv"

# Step 1: Load existing scraped data
existing_data = {}
if os.path.exists(output_csv):
    with open(output_csv, newline='', encoding='utf-8') as existing_file:
        reader = csv.DictReader(existing_file)
        for row in reader:
            existing_data[row['competition_id']] = {
                'competition_name': row['competition_name'],
                'cup_image_url': row.get('cup_image_url', '').strip(),
                'competition_logo_url': row.get('competition_logo_url', '').strip()
            }

# Step 2: Prepare output file (temporary storage)
output_rows = []

# Step 3: Read input and decide what needs scraping
with open(input_csv, newline='', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)

    for row in reader:
        comp_id = row['competition_id'].strip()
        comp_name = row['name'].strip()
        url = row['url'].strip()

        existing = existing_data.get(comp_id, {})
        cup_exists = bool(existing.get('cup_image_url'))
        logo_exists = bool(existing.get('competition_logo_url'))

        if cup_exists and logo_exists:
            print(f"⏭️ Skipping {comp_name} ({comp_id}) — already has both images")
            output_rows.append({
                'competition_id': comp_id,
                'competition_name': comp_name,
                'cup_image_url': existing['cup_image_url'],
                'competition_logo_url': existing['competition_logo_url']
            })
            continue

        print(f"🔍 Scraping {comp_name} ({comp_id})...")
        cup_url = existing.get('cup_image_url', '')
        logo_url = existing.get('competition_logo_url', '')

        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=30)

            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                def full_url(src):
                    if src.startswith("//"):
                        return "https:" + src
                    elif src.startswith("/"):
                        return "https://www.transfermarkt.com" + src
                    return src

                if not cup_exists:
                    cup_img_tag = soup.find('img', src=lambda src: src and "/images/erfolge/fix/" in src)
                    cup_url = full_url(cup_img_tag['src']) if cup_img_tag else ''

                if not logo_exists:
                    logo_img_tag = soup.find('img', src=lambda src: src and "/images/logo/header/" in src)
                    logo_url = full_url(logo_img_tag['src']) if logo_img_tag else ''

                print(f"✅ Cup: {cup_url if cup_url else '❌ Not found'} | Logo: {logo_url if logo_url else '❌ Not found'}")

            else:
                print(f"❌ Failed to fetch {url} (status {response.status_code})")

        except Exception as e:
            print(f"❌ Error scraping {comp_name}: {e}")

        output_rows.append({
            'competition_id': comp_id,
            'competition_name': comp_name,
            'cup_image_url': cup_url,
            'competition_logo_url': logo_url
        })

        time.sleep(1)  # polite delay

# Step 4: Write full output back (overwrite)
with open(output_csv, mode='w', newline='', encoding='utf-8') as outfile:
    fieldnames = ['competition_id', 'competition_name', 'cup_image_url', 'competition_logo_url']
    writer = csv.DictWriter(outfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(output_rows)

print("✅ Done! All scraped data saved.")


⏭️ Skipping italy-cup (CIT) — already has both images
⏭️ Skipping johan-cruijff-schaal (NLSC) — already has both images
⏭️ Skipping kypello-elladas (GRP) — already has both images
⏭️ Skipping supertaca-candido-de-oliveira (POSU) — already has both images
⏭️ Skipping russian-super-cup (RUSS) — already has both images
⏭️ Skipping supercopa (SUC) — already has both images
⏭️ Skipping uefa-super-cup (USC) — already has both images
🔍 Scraping superligaen (DK1)...
✅ Cup: ❌ Not found | Logo: https://tmssl.akamaized.net//images/logo/header/dk1.png?lm=1739371406
⏭️ Skipping europa-league (EL) — already has both images
🔍 Scraping laliga (ES1)...
✅ Cup: ❌ Not found | Logo: https://tmssl.akamaized.net//images/logo/header/es1.png?lm=1725974302
🔍 Scraping ligue-1 (FR1)...
✅ Cup: ❌ Not found | Logo: https://tmssl.akamaized.net//images/logo/header/fr1.png?lm=1732280518
🔍 Scraping serie-a (IT1)...
✅ Cup: ❌ Not found | Logo: https://tmssl.akamaized.net//images/logo/header/it1.png?lm=1656073460
🔍 Scrapin